# DA3 FPS Pipeline (Notebook)

This notebook does the full pipeline:

1. Set an FPS variable
2. Split video into frames by FPS
3. Run DA3 CLI inference on each frame
4. Gather all inferred images into one directory
5. Rebuild video and keep original audio


In [ ]:
from IPython.display import clear_output

# 克隆DA3官方仓库
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3.git
%cd Depth-Anything-3

# 安装基础依赖
!pip install xformers 

# 按照文档以可编辑模式安装DA3
!pip install -e .
# !pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
%pip install gsplat

# %% [code]
# 精确替换错误的变量名，修复官方 cli.py 中的 Bug
# !sed -i "s/reference_view_strategy=reference_view_strategy/ref_view_strategy=ref_view_strategy/g" /kaggle/working/Depth-Anything-3/src/depth_anything_3/cli.py
!find /kaggle/working/Depth-Anything-3/src -type f -name "*.py" -exec sed -i 's/reference_view_strategy/ref_view_strategy/g' {} +
# 确认替换成功（可选，用于打印修改后的那行代码查看）
!grep "ref_view_strategy" /kaggle/working/Depth-Anything-3/src/depth_anything_3/cli.py
clear_output()

In [ ]:
from pathlib import Path
import shutil
import subprocess

# ====== User Config ======
INPUT_VIDEO = Path("/kaggle/input/datasets/liuweiq/daxiaonailong/caixunkun.mp4")
FPS = None  # None means use source video fps
MODEL_DIR = "depth-anything/DA3-SMALL"
PROCESS_RES = 504
PROCESS_RES_METHOD = "upper_bound_resize"
USE_BACKEND = False
BACKEND_URL = "http://127.0.0.1:8008"

WORK_ROOT = Path("/kaggle/working/da3_fps_pipeline")
FRAMES_DIR = WORK_ROOT / "frames"
INFER_DIR = WORK_ROOT / "infer_raw"
COLLECTED_DIR = WORK_ROOT / "collected_images"
OUTPUT_VIDEO = WORK_ROOT / "depth_with_audio.mp4"

print("INPUT_VIDEO:", INPUT_VIDEO)
print("FPS (requested):", FPS)
print("MODEL_DIR:", MODEL_DIR)
print("WORK_ROOT:", WORK_ROOT)

In [ ]:
def run_cmd(cmd):
    cmd = [str(c) for c in cmd]
    print("+", " ".join(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.returncode != 0:
        print("\n===== COMMAND FAILED =====")
        print("Return code:", result.returncode)
        print("\n----- STDOUT -----")
        print(result.stdout if result.stdout else "<empty>")
        print("\n----- STDERR -----")
        print(result.stderr if result.stderr else "<empty>")
        raise RuntimeError("Command failed, see stdout/stderr above.")
    return result


def get_video_fps(video_path):
    out = subprocess.check_output(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            "stream=r_frame_rate",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            video_path,
        ],
        text=True,
    ).strip()

    if "/" in out:
        num, den = out.split("/", 1)
        return float(num) / float(den)
    return float(out)


if not INPUT_VIDEO.exists():
    raise FileNotFoundError(f"Input video not found: {INPUT_VIDEO}")

if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg is not found in PATH")

if shutil.which("ffprobe") is None:
    raise RuntimeError("ffprobe is not found in PATH")

if shutil.which("da3") is None:
    raise RuntimeError("da3 CLI is not found in PATH. Install Depth-Anything-3 first.")

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

FRAMES_DIR.mkdir(parents=True, exist_ok=True)
INFER_DIR.mkdir(parents=True, exist_ok=True)
COLLECTED_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FPS = get_video_fps(str(INPUT_VIDEO))
EFFECTIVE_FPS = SOURCE_FPS if FPS in (None, "", 0, 0.0) else float(FPS)

print("Environment check passed.")
print(f"Source FPS: {SOURCE_FPS:.6f}")
print(f"Effective FPS: {EFFECTIVE_FPS:.6f}")

In [ ]:
# 1) Split video into frames by FPS (default: source fps)
frame_pattern = FRAMES_DIR / "frame_%06d.png"

split_cmd = [
    "ffmpeg",
    "-y",
    "-i",
    INPUT_VIDEO,
]

if FPS not in (None, "", 0, 0.0):
    split_cmd.extend(["-vf", f"fps={EFFECTIVE_FPS}"])

split_cmd.append(frame_pattern)
run_cmd(split_cmd)

frame_list = sorted(FRAMES_DIR.glob("*.png"))
print(f"Extracted frames: {len(frame_list)}")
if not frame_list:
    raise RuntimeError("No frames were extracted.")

In [ ]:
# 2) Run DA3 CLI inference frame-by-frame (da3 image)
if INFER_DIR.exists():
    shutil.rmtree(INFER_DIR)
INFER_DIR.mkdir(parents=True, exist_ok=True)

for idx, frame_path in enumerate(frame_list, start=1):
    sample_dir = INFER_DIR / frame_path.stem
    sample_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "da3",
        "image",
        frame_path,
        "--model-dir",
        MODEL_DIR,
        "--export-dir",
        sample_dir,
        "--export-format",
        "glb",
        "--process-res",
        str(PROCESS_RES),
        "--process-res-method",
        PROCESS_RES_METHOD,
        "--auto-cleanup",
    ]

    if USE_BACKEND:
        cmd.extend(["--use-backend", "--backend-url", BACKEND_URL])

    print(f"[{idx}/{len(frame_list)}] infer: {frame_path.name}")
    run_cmd(cmd)

print("Inference finished.")

In [ ]:
# 3) Gather inferred images into one folder
if COLLECTED_DIR.exists():
    shutil.rmtree(COLLECTED_DIR)
COLLECTED_DIR.mkdir(parents=True, exist_ok=True)

collected = 0
for idx, frame_path in enumerate(frame_list, start=1):
    sample_dir = INFER_DIR / frame_path.stem
    scene_jpg = sample_dir / "scene.jpg"

    if not scene_jpg.exists():
        candidates = sorted(sample_dir.rglob("*.jpg")) + sorted(
            sample_dir.rglob("*.png")
        )
        if not candidates:
            raise FileNotFoundError(f"No rendered image found in {sample_dir}")
        scene_jpg = candidates[0]

    out_name = f"frame_{idx:06d}.jpg"
    shutil.copy2(scene_jpg, COLLECTED_DIR / out_name)
    collected += 1

print(f"Collected images: {collected}")
if collected == 0:
    raise RuntimeError("No inferred images collected.")

In [ ]:
# 4) Rebuild video from collected images and keep original audio
render_pattern = COLLECTED_DIR / "frame_%06d.jpg"
run_cmd(
    [
        "ffmpeg",
        "-y",
        "-framerate",
        str(EFFECTIVE_FPS),
        "-i",
        render_pattern,
        "-i",
        INPUT_VIDEO,
        "-map",
        "0:v:0",
        "-map",
        "1:a:0?",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-shortest",
        OUTPUT_VIDEO,
    ]
)

print("Output video:", OUTPUT_VIDEO)

In [ ]:
# 5) Quick summary
print("Frames dir:", FRAMES_DIR)
print("Raw infer dir:", INFER_DIR)
print("Collected image dir:", COLLECTED_DIR)
print("Source FPS:", SOURCE_FPS)
print("Effective FPS:", EFFECTIVE_FPS)
print("Final video:", OUTPUT_VIDEO)
print("Final video exists:", OUTPUT_VIDEO.exists())